V11 Мат модель для нахождения коэффициента влияния гендера на ЗП

In [7]:
import pandas as pd

modelv1 = pd.read_excel("../../data/processed/v1_clean_base.xlsx")

model v11 (логарифмическая регрессия по параметрам: salary, gender, experience, education_type, region_code, industry_code)

In [8]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import warnings

warnings.filterwarnings("ignore")


df = modelv1.copy()

df["salary"] = pd.to_numeric(df["salary"], errors="coerce")
df["experience"] = pd.to_numeric(df["experience"], errors="coerce").fillna(0)

df["region_code"] = (
    df["region_code"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D+", "", regex=True)
    .str.zfill(13)
    .str[:2]
)
df["region_code"] = pd.to_numeric(df["region_code"], errors="coerce").astype("Int64")

df = df[
    (df["salary"] > 5000) &
    (df["salary"] < 165000) &
    (df["gender"].isin(["Мужской", "Женский"])) &
    (df["industry_code"] == "BuldindRealty") &
    (df["region_code"].notna())
].copy()

df["male"] = (df["gender"] == "Мужской").astype(int)
df = df[df["salary"] > 0].copy()
df["log_salary"] = np.log(df["salary"])

results = []

for code, grp in df.groupby("region_code"):
    nm = int((grp["gender"] == "Мужской").sum())
    nf = int((grp["gender"] == "Женский").sum())

    if nm < 5 or nf < 5:
        continue

    reg = smf.ols("log_salary ~ male + experience", data=grp).fit(cov_type="HC3")

    gender_pct = (np.exp(reg.params["male"]) - 1) * 100
    exp_pct = (np.exp(reg.params["experience"]) - 1) * 100

    results.append({
        "регион": int(code),
        "n_М": nm,
        "n_Ж": nf,
        "n_всего": len(grp),
        "коэф_gender": round(reg.params["male"], 6),
        "gender_эффект_%": round(gender_pct, 3),
        "p_gender": round(reg.pvalues["male"], 6),
        "gender_значимо": "да" if reg.pvalues["male"] < 0.05 else "нет",
        "коэф_опыт": round(reg.params["experience"], 6),
        "опыт_эффект_%_за_год": round(exp_pct, 3),
        "p_опыт": round(reg.pvalues["experience"], 6),
        "R2": round(reg.rsquared, 4)
    })

v11_result = pd.DataFrame(results).sort_values(["p_gender", "регион"]).reset_index(drop=True)

print(v11_result.to_string(index=False))

 регион  n_М  n_Ж  n_всего  коэф_gender  gender_эффект_%  p_gender gender_значимо  коэф_опыт  опыт_эффект_%_за_год  p_опыт     R2
      0  597  430     1027     0.220975           24.729       0.0             да   0.020151                 2.036     0.0 0.1463


Сохранение результатов model v11

In [9]:
v11_result.to_excel("../../data/processed/v11_result.xlsx", index=False)